# GitHub API / CLI 全流程验证

> **目标**：验证 GitHub REST API 的完整能力链路,为 Agent 工具开发提供依据
> **覆盖**：认证 → 仓库 → 内容 → Issue → PR → 搜索 → 提交 → 工作流
> **认证**：`GITHUB_TOKEN`(Personal Access Token)

每个 API 调用旁标注等效的 `gh` CLI 命令,供对比参考. 

In [ ]:
from github_client import GitHubClient, format_repo_info, format_issue, format_pull, format_commit
from pathlib import Path
import json

# 初始化客户端(自动从 .env 读取 GITHUB_TOKEN)
client = GitHubClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] GitHubClient initialized")
if health['ok']:
    print(f"       User: @{health['login']}")
else:
    print(f"       Error: {health.get('error')}")
    print(f"       请检查 .env 中的 GITHUB_TOKEN 是否有效")

[OK] GitHubClient initialized
       User: @SingularGuyLeBorn


## 1. 仓库操作

验证 `GET /repos/{owner}/{repo}` 和搜索能力. 

In [ ]:
# 1.1 获取仓库信息
# gh> gh repo view facebook/react --json name,description,stargazersCount
repo = client.get_repo("facebook", "react")
print(format_repo_info(repo))

📦 facebook/react
   ⭐ 244,665  🍴 50,979  🐛 1,243
   📝 The library for web and native user interfaces.
   🔤 主要语言: JavaScript
   🕐 更新于: 2026-04-24T23:27:40Z
   🔗 https://github.com/facebook/react


In [ ]:
# 1.2 搜索仓库
# gh> gh search repos "react state management" --limit 5
result = client.search_repos("react state management", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个仓库 (显示 {len(result.get('items', []))} 个)\n")
for item in result.get("items", [])[:5]:
    print(f"  {item['full_name']} ⭐ {item['stargazers_count']:,} — {item.get('description', '')[:60]}")

找到 25315 个仓库 (显示 5 个)

  mobxjs/mobx ⭐ 28,183 — Simple, scalable state management.
  jamiebuilds/unstated ⭐ 7,738 — State so simple, it goes without saying
  pmndrs/zustand ⭐ 57,849 — 🐻 Bear necessities for state management in React
  pmndrs/jotai ⭐ 21,129 — 👻 Primitive and flexible state management for React
  react-hook-form/react-hook-form ⭐ 44,676 — 📋 React Hooks for form state management and validation (Web 


## 2. 内容读取

验证 `GET /repos/{owner}/{repo}/contents/{path}` —— 目录列表和文件内容提取. 

In [ ]:
# 2.1 列出根目录内容
# gh> gh api repos/facebook/react/contents/
contents = client.list_contents("facebook", "react", "")
print(f"根目录共 {len(contents)} 项:\n")
for item in contents[:10]:
    icon = "📁" if item['type'] == 'dir' else "📄"
    print(f"  {icon} {item['name']}")

根目录共 36 项:

  📁 .claude
  📁 .codesandbox
  📄 .editorconfig
  📄 .eslintignore
  📄 .eslintrc.js
  📄 .git-blame-ignore-revs
  📄 .gitattributes
  📁 .github
  📄 .gitignore
  📄 .mailmap


In [ ]:
# 2.2 读取 README.md 内容
# gh> gh api repos/facebook/react/contents/README.md | jq -r .content | base64 -d
readme = client.get_file_content("facebook", "react", "README.md")
print(readme[:2000])
print(f"\n... (共 {len(readme)} 字符)")

# [React](https://react.dev/) &middot; [![GitHub license](https://img.shields.io/badge/license-MIT-blue.svg)](https://github.com/facebook/react/blob/main/LICENSE) [![npm version](https://img.shields.io/npm/v/react.svg?style=flat)](https://www.npmjs.com/package/react) [![(Runtime) Build and Test](https://github.com/facebook/react/actions/workflows/runtime_build_and_test.yml/badge.svg)](https://github.com/facebook/react/actions/workflows/runtime_build_and_test.yml) [![(Compiler) TypeScript](https://github.com/facebook/react/actions/workflows/compiler_typescript.yml/badge.svg?branch=main)](https://github.com/facebook/react/actions/workflows/compiler_typescript.yml) [![PRs Welcome](https://img.shields.io/badge/PRs-welcome-brightgreen.svg)](https://legacy.reactjs.org/docs/how-to-contribute.html#your-first-pull-request)

React is a JavaScript library for building user interfaces.

* **Declarative:** React makes it painless to create interactive UIs. Design simple views for each state in your

In [ ]:
# 2.3 读取 src 目录下的文件列表
src_contents = client.list_contents("facebook", "react", "packages/react/src")
print(f"packages/react/src 共 {len(src_contents)} 项:\n")
for item in src_contents:
    print(f"  {'📁' if item['type'] == 'dir' else '📄'} {item['name']}")

packages/react/src 共 29 项:

  📄 BadMapPolyfill.js
  📄 ReactAct.js
  📄 ReactBaseClasses.js
  📄 ReactCacheClient.js
  📄 ReactCacheImpl.js
  📄 ReactCacheServer.js
  📄 ReactChildren.js
  📄 ReactClient.js
  📄 ReactCompilerRuntime.js
  📄 ReactContext.js
  📄 ReactCreateRef.js
  📄 ReactForwardRef.js
  📄 ReactHooks.js
  📄 ReactLazy.js
  📄 ReactMemo.js
  📄 ReactNoopUpdateQueue.js
  📄 ReactOwnerStack.js
  📄 ReactServer.experimental.development.js
  📄 ReactServer.experimental.js
  📄 ReactServer.fb.js
  📄 ReactServer.js
  📄 ReactSharedInternalsClient.js
  📄 ReactSharedInternalsServer.js
  📄 ReactStartTransition.js
  📄 ReactTaint.js
  📄 ReactTaintRegistry.js
  📄 ReactTransitionType.js
  📁 __tests__
  📁 jsx


## 3. Issue 操作

验证 `GET /repos/{owner}/{repo}/issues` 和 `POST /repos/{owner}/{repo}/issues`. 

In [ ]:
# 3.1 列出 open issues
# gh> gh issue list --repo facebook/react --state open --limit 5
issues = client.list_issues("facebook", "react", state="open", per_page=5)
print(f"Open Issues ({len(issues)} 条):\n")
for issue in issues[:5]:
    print(f"  {format_issue(issue)}")

Open Issues (5 条):

  #36343 [open] fix: handle null-prototype objects in typeName helper — @srpatcha
  #36342 [open] Bump postcss from 8.4.47 to 8.5.10 in /compiler/apps/playground — @dependabot[bot]
  #36341 [open] README and CONTRIBUTING.md use inconsistent legacy contribution links — @ayangabryl
  #36340 [open] Warn when NaN is passed as an aria-* attribute value — @starboyvarun
  #36339 [open] fix(react-devtools-fusebox): change private field from string to boolean — @Jah-yee


In [ ]:
# 3.2 获取单个 Issue 详情
# gh> gh issue view 1 --repo facebook/react
if issues:
    first = issues[0]
    detail = client.get_issue("facebook", "react", first['number'])
    print(f"\nIssue #{detail['number']}: {detail['title']}\n")
    print(detail.get('body', '无内容')[:500])
    print(f"\n  Labels: {[l['name'] for l in detail.get('labels', [])]}")
    print(f"  Comments: {detail.get('comments', 0)}")


Issue #36343: fix: handle null-prototype objects in typeName helper

## Summary\n\nFixes crash in packages/shared/CheckStringCoercion.js:\n\n\	ypeName()\ accessed \alue.constructor.name\ without guarding against null-prototype objects (created via \Object.create(null)\), which have no \.constructor\ property. This prevented helpful error messages from printing.\n\nAdded guard with fallback to \Object.prototype.toString.call()\ and try-catch wrapper.\n\nAlso adds .gitattributes for consistent line endings.\n\nSigned-off-by: Srikanth Patchava <spatchava@meta.com>

  Labels: ['CLA Signed']
  Comments: 0


## 4. Pull Request 操作

验证 `GET /repos/{owner}/{repo}/pulls` —— PR 列表和详情. 

In [ ]:
# 4.1 列出 open PRs
# gh> gh pr list --repo facebook/react --state open --limit 5
pulls = client.list_pulls("facebook", "react", state="open", per_page=5)
print(f"Open PRs ({len(pulls)} 条):\n")
for pr in pulls[:5]:
    print(f"  {format_pull(pr)}")

Open PRs (5 条):

  #36343 [open] fix: handle null-prototype objects in typeName helper — @srpatcha
  #36342 [open] Bump postcss from 8.4.47 to 8.5.10 in /compiler/apps/playground — @dependabot[bot]
  #36340 [open] Warn when NaN is passed as an aria-* attribute value — @starboyvarun
  #36339 [open] fix(react-devtools-fusebox): change private field from string to boolean — @Jah-yee
  #36337 [open] Fix false positive for self-referential useCallback — @KimHyeongRae0


In [ ]:
# 4.2 获取单个 PR 详情
# gh> gh pr view 123 --repo facebook/react
if pulls:
    first = pulls[0]
    detail = client.get_pull("facebook", "react", first['number'])
    print(f"\nPR #{detail['number']}: {detail['title']}\n")
    print(f"  Author: @{detail['user']['login']}")
    print(f"  Branch: {detail['head']['ref']} → {detail['base']['ref']}")
    print(f"  Status: {detail['state']} | Draft: {detail.get('draft', False)}")
    print(f"  Commits: {detail.get('commits', 0)} | Additions: +{detail.get('additions', 0)} | Deletions: -{detail.get('deletions', 0)}")


PR #36343: fix: handle null-prototype objects in typeName helper

  Author: @srpatcha
  Branch: chore/add-gitattributes → main
  Status: open | Draft: False
  Commits: 4 | Additions: +587 | Deletions: -9


## 5. 代码搜索

验证 `GET /search/code` —— 跨仓库代码搜索. 

In [ ]:
# 5.1 搜索代码
# gh> gh search code "useState hook" language:typescript --limit 5
result = client.search_code("useState hook language:typescript", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个结果 (显示 {len(result.get('items', []))} 个)\n")
for item in result.get("items", [])[:5]:
    print(f"  📄 {item['repository']['full_name']}: {item['path']}")
    print(f"     {item['html_url']}")

找到 353792 个结果 (显示 5 个)

  📄 frejs/fre: src/hook.ts
     https://github.com/frejs/fre/blob/e1c4b655d03956ead9b1ea40ec747cdb53b3ba39/src/hook.ts
  📄 littledivy/wgui: hooks.ts
     https://github.com/littledivy/wgui/blob/785182dc08fb8b5a1fd0e835518f4142e3d251cf/hooks.ts
  📄 WebReflection/neverland: index.d.ts
     https://github.com/WebReflection/neverland/blob/894a4147c384b95e7cfa1f0f44b330ad3a03ef06/index.d.ts
  📄 seatedro/act: index.ts
     https://github.com/seatedro/act/blob/36127fc4d2d0e3853fee62c81bd26afcdafa2e70/index.ts
  📄 shenAwesome/relax: src/relax.ts
     https://github.com/shenAwesome/relax/blob/cf9c3370c3ba3a2cac8b18d08349a76b933df993/src/relax.ts


In [ ]:
# 5.2 搜索 Issues
# gh> gh search issues "memory leak" --limit 5
result = client.search_issues("memory leak", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个 issues (显示 {len(result.get('items', []))} 个)
")
for item in result.get("items", [])[:5]:
    print(f"  #{item['number']} [{item['state']}] {item['title'][:60]}  @{item['user']['login']}")


找到 272 个结果 (显示 5 个)

  #36223 [open] Call stopTracking() on host component unmount to prevent inp — @stewartmcgown
  #16087 [open] [Umbrella] Memory Leaks — @sebmarkbage
  #35670 [open] fix: properly remove event listeners in component unmount to — @ZIRUL-0902
  #35814 [open] Bug: Memory leak in attribute-behavior fixture - wrong varia — @AmitSingh-5600
  #14962 [open] Password input type causes memory leak — @ejallday


## 6. 提交历史

验证 `GET /repos/{owner}/{repo}/commits` —— 提交记录和文件级历史. 

In [ ]:
# 6.1 最近提交
# gh> gh api repos/facebook/react/commits?per_page=5
commits = client.get_commits("facebook", "react", per_page=5)
print(f"最近 {len(commits)} 条提交:\n")
for c in commits:
    print(f"  {format_commit(c)}")

最近 5 条提交:

  561ed52 Fix formatting (#36332) — Sebastian "Sebbie" Silbermann
  142cfde Fix FragmentInstance listener leak: normalize boolean vs obj — Kotha Dhakshin
  94643c3 Suggest correct casing for misspelled credentialless iframe  — vmx906
  306a01b Add credentialless as a recognized boolean attribute for ifr — vmx906
  3ee1fe4 Fix contributor attribution for ESLint v10 support — Jack Pope


In [ ]:
# 6.2 特定文件的提交历史
# gh> gh api repos/facebook/react/commits?path=packages/react/src/ReactHooks.js&per_page=5
file_commits = client.get_commits("facebook", "react", path="packages/react/src/ReactHooks.js", per_page=5)
print(f"packages/react/src/ReactHooks.js 最近 {len(file_commits)} 条提交:\n")
for c in file_commits:
    print(f"  {format_commit(c)}")

packages/react/src/ReactHooks.js 最近 5 条提交:

  0a7cf20 Remove useSwipeTransition (#32786) — Sebastian Markbåge
  313332d [crud] Revert CRUD overload (#32741) — lauren
  a53da6a Add useSwipeTransition Hook Behind Experimental Flag (#32373 — Sebastian Markbåge
  192555b Added dev-only warning for null/undefined create in use*Effe — Josh Goldberg ✨
  a69b80d [crud] Remove useResourceEffect (#32206) — lauren


## 7. Actions 工作流

验证 `GET /repos/{owner}/{repo}/actions/workflows` 和运行记录. 

In [ ]:
# 7.1 列出工作流
# gh> gh workflow list --repo facebook/react
workflows = client.list_workflows("facebook", "react")
print(f"工作流 ({workflows.get('total_count', 0)} 个):\n")
for wf in workflows.get("workflows", [])[:5]:
    state = "✅" if wf['state'] == 'active' else "⏸️"
    print(f"  {state} {wf['name']} (id={wf['id']}, path={wf['path']})")

工作流 (29 个):

  ✅ (DevTools) Regression Tests (id=94341022, path=.github/workflows/devtools_regression_tests.yml)
  ✅ (Compiler) TypeScript (id=103880090, path=.github/workflows/compiler_typescript.yml)
  ✅ (Shared) Lint (id=103880092, path=.github/workflows/shared_lint.yml)
  ✅ React Compiler (Rust) (id=103880095, path=.github/workflows/compiler_rust.yml)
  ✅ (Compiler) Playground (id=103880096, path=.github/workflows/compiler_playground.yml)


In [ ]:
# 7.2 最近运行记录
# gh> gh run list --repo facebook/react --limit 5
runs = client.list_workflow_runs("facebook", "react", per_page=5)
print(f"最近运行 ({runs.get('total_count', 0)} 次):\n")
for run in runs.get("workflow_runs", [])[:5]:
    status_icon = "✅" if run['conclusion'] == 'success' else "❌" if run['conclusion'] == 'failure' else "⏳"
    print(f"  {status_icon} {run['name']} — {run['head_branch']} @ {run['run_number']}")
    print(f"     状态: {run['status']} | 结论: {run['conclusion'] or 'N/A'} | 触发: {run['event']}")
    print(f"     {run['html_url']}")

最近运行 (40000 次):

  ✅ (Runtime) Fuzz tests — main @ 18253
     状态: completed | 结论: success | 触发: schedule
     https://github.com/facebook/react/actions/runs/24932066198
  ✅ (Shared) Manage stale issues and PRs — main @ 15877
     状态: completed | 结论: success | 触发: schedule
     https://github.com/facebook/react/actions/runs/24932032780
  ✅ (Runtime) Fuzz tests — main @ 18252
     状态: completed | 结论: success | 触发: schedule
     https://github.com/facebook/react/actions/runs/24930787703
  ✅ (Shared) Manage stale issues and PRs — main @ 15876
     状态: completed | 结论: success | 触发: schedule
     https://github.com/facebook/react/actions/runs/24930714885
  ✅ (Shared) Cleanup Stale Branch Caches — main @ 1568
     状态: completed | 结论: success | 触发: schedule
     https://github.com/facebook/react/actions/runs/24930511352


## 9. 完整写入流程测试(⚠️ 会真实修改仓库)

> **完整链路**：创建仓库 → Issue 模板 → Issue CRUD → 分支 → Commit → PR → Review → Merge
>
> 运行前确保 Token 有 `repo` + `workflow` scope. 
> 所有变量在 cells 之间共享,请**按顺序执行**. 

In [ ]:
# 9.0 配置
TEST_OWNER = "SingularGuyLeBorn"
TEST_REPO = "github-api-lab45"

print(f"测试账号: {TEST_OWNER}")
print(f"测试仓库: {TEST_REPO}")
print("⚠️  以下操作会真实修改仓库数据,请确认配置正确！")

测试账号: SingularGuyLeBorn
测试仓库: github-api-lab45
⚠️  以下操作会真实修改仓库数据,请确认配置正确！


In [ ]:
# 9.1 创建仓库(如果已存在会报错 422,可忽略或手动删除后重试)
# gh> gh repo create github-api-lab --public --add-readme
try:
    repo_info = client.create_repo(
        name=TEST_REPO,
        description="GitHub API 写入测试专用仓库2",
        private=False,
        auto_init=True
    )
    print(f"✅ 仓库创建成功: {repo_info['html_url']}")
except Exception as e:
    if "422" in str(e):
        print("⚠️  仓库已存在,跳过创建")
    else:
        raise

✅ 仓库创建成功: https://github.com/SingularGuyLeBorn/github-api-lab45


In [ ]:
# 9.2 设置 Issue 模板
# Issue 模板是仓库文件: .github/ISSUE_TEMPLATE/bug_report.md
template_content = '''---
name: Bug Report
about: 使用 API 测试创建的模板
title: '[Bug] '
labels: bug
---

**描述问题**
清晰简洁地描述 bug. 
'''

template_result = client.create_or_update_file(
    TEST_OWNER, TEST_REPO,
    path=".github/ISSUE_TEMPLATE/bug_report.md",
    content=template_content,
    message="[test] add issue template",
    branch="main"
)
print(f"✅ Issue 模板创建成功: {template_result['content']['path']}")

✅ Issue 模板创建成功: .github/ISSUE_TEMPLATE/bug_report.md


In [ ]:
# 9.3 创建两个 Issue
# Issue A：完整 CRUD,最终关闭
# Issue B：验证关闭后再打开,最终保留 open

issue_a = client.create_issue(
    TEST_OWNER, TEST_REPO,
    title="[Test] 将被关闭的 Issue A",
    body="这条 Issue 会被操作(评论、更新),最终关闭. ",
    labels=["bug"]
)
issue_a_number = issue_a['number']
print(f"✅ Issue A 创建成功: #{issue_a_number}")

issue_b = client.create_issue(
    TEST_OWNER, TEST_REPO,
    title="[Test] 关闭后重开的 Issue B",
    body="这条 Issue 会经历关闭→重开,最终保留 open. ",
    labels=["enhancement"]
)
issue_b_number = issue_b['number']
print(f"✅ Issue B 创建成功: #{issue_b_number}")

✅ Issue A 创建成功: #1
✅ Issue B 创建成功: #2


In [ ]:
# 9.4 读取 Issue A
issue_detail = client.get_issue(TEST_OWNER, TEST_REPO, issue_a_number)
print(f"Issue #{issue_detail['number']}: {issue_detail['title']}")
print(f"状态: {issue_detail['state']}")
print(f"标签: {[l['name'] for l in issue_detail.get('labels', [])]}")
print(f"评论数: {issue_detail.get('comments', 0)}")

Issue #1: [Test] 将被关闭的 Issue A
状态: open
标签: ['bug']
评论数: 0


In [ ]:
# 9.5 回答 Issue A(添加评论)
comment = client.create_issue_comment(
    TEST_OWNER, TEST_REPO, issue_a_number,
    body="第一步：验证 create_issue_comment 正常. "
)
comment_id = comment['id']
print(f"✅ 评论添加成功: ID={comment_id}")

comment2 = client.create_issue_comment(
    TEST_OWNER, TEST_REPO, issue_a_number,
    body="第二步：验证多条评论支持. "
)
comment2_id = comment2['id']
print(f"✅ 第二条评论添加成功: ID={comment2_id}")

✅ 评论添加成功: ID=4303329504
✅ 第二条评论添加成功: ID=4303329654


In [ ]:
# 9.6 更新 Issue A
updated = client.update_issue(
    TEST_OWNER, TEST_REPO, issue_a_number,
    title="[Test] 完整流程验证 Issue A [已更新]",
    body="Issue 已被更新：修改了标题和正文内容. ",
    labels=["bug", "enhancement"]
)
print(f"✅ Issue 更新成功: {updated['title']}")
print(f"   新标签: {[l['name'] for l in updated.get('labels', [])]}")

✅ Issue 更新成功: [Test] 完整流程验证 Issue A [已更新]
   新标签: ['bug', 'enhancement']


In [ ]:
# 9.7 关闭 Issue A(保持 closed,验证关闭功能)
client.update_issue(TEST_OWNER, TEST_REPO, issue_a_number, state="closed")
print(f"✅ Issue #{issue_a_number} 已关闭")

✅ Issue #1 已关闭


In [ ]:
# 9.8 Issue B：先关闭,再重开(验证两个功能)
client.update_issue(TEST_OWNER, TEST_REPO, issue_b_number, state="closed")
print(f"✅ Issue #{issue_b_number} 已关闭")

client.update_issue(TEST_OWNER, TEST_REPO, issue_b_number, state="open")
print(f"✅ Issue #{issue_b_number} 已重新打开")

✅ Issue #2 已关闭
✅ Issue #2 已重新打开


In [ ]:
# 9.8b Issue B 添加复杂格式评论(图文并茂、表格、链接、代码块)

comment_b1 = client.create_issue_comment(
    TEST_OWNER, TEST_REPO, issue_b_number,
    body=f"""## 📋 API 测试报告

这是一份由 **`github_client.py`** 自动生成的复杂格式评论. 

### ✅ 功能检查清单

| 功能 | 状态 | 备注 |
|------|:----:|------|
| 创建 Issue | ✅ 通过 | 编号 #{issue_b_number} |
| 添加评论 | ✅ 通过 | 支持 Markdown 全格式 |
| 更新 Issue | ⏳ 待测 | 后续验证 |
| 关闭/重开 | 🔄 已测 | Cell 9.8 |

### 🔗 相关链接

- [GitHub REST API 文档](https://docs.github.com/en/rest)
- [GitHub Flavored Markdown 规范](https://github.github.com/gfm/)

### 💻 代码示例

```python
from github_client import GitHubClient

client = GitHubClient()
client.create_issue("owner", "repo", title="Hello", body="World")
```

> 💡 **提示**：这条评论完全通过 REST API 提交,无需本地 git 操作. 

### 📌 任务列表

- [x] 支持多级标题
- [x] 支持表格
- [x] 支持代码块
- [x] 支持链接
- [ ] 上传本地图片(需先提交到仓库再引用)
"""
)
print(f"✅ 评论 1 添加成功: ID={comment_b1['id']}")

comment_b2 = client.create_issue_comment(
    TEST_OWNER, TEST_REPO, issue_b_number,
    body="""### 🖼️ 图文混排测试

**加粗** 和 *斜体* 和 ~~删除线~~ 样式. 

1. 第一步：创建 Issue
2. 第二步：添加评论
   - 子项 A：纯文本
   - 子项 B：富文本
3. 第三步：验证渲染

![GitHub Mark](https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png)

---

**引用分隔线之上的内容**

> 这是一段引用块,用于测试样式渲染. 
"""
)
print(f"✅ 评论 2 添加成功: ID={comment_b2['id']}")


✅ 评论 1 添加成功: ID=4303330217
✅ 评论 2 添加成功: ID=4303330386


In [ ]:
# 9.9 创建两个分支(A 用于 PR,B 保留)
branch_a = "feature-a"
branch_b = "feature-b"

client.create_branch(TEST_OWNER, TEST_REPO, branch=branch_a, from_branch="main")
print(f"✅ 分支 A 创建成功: {branch_a}")

client.create_branch(TEST_OWNER, TEST_REPO, branch=branch_b, from_branch="main")
print(f"✅ 分支 B 创建成功: {branch_b}(保留,不删除)")

✅ 分支 A 创建成功: feature-a
✅ 分支 B 创建成功: feature-b(保留,不删除)


In [ ]:
# 9.10 在 feature-a 提交文件
file_path = "test-feature.md"
file_content = "# Test Feature\n\nThis file is created by API test.\n"

file_result = client.create_or_update_file(
    TEST_OWNER, TEST_REPO,
    path=file_path,
    content=file_content,
    message="[test] add test-feature.md",
    branch=branch_a
)
file_sha = file_result['content']['sha']
print(f"✅ 文件提交成功: {file_path} (sha: {file_sha[:7]})")

✅ 文件提交成功: test-feature.md (sha: 74c3d9b)


In [ ]:
# 9.11 更新文件(产生新的 commit)
updated_content = "# Test Feature\n\nThis file has been updated by API test.\n\n- [x] Create\n- [x] Update\n"

update_result = client.create_or_update_file(
    TEST_OWNER, TEST_REPO,
    path=file_path,
    content=updated_content,
    message="[test] update test-feature.md",
    branch=branch_a,
    sha=file_sha
)
new_file_sha = update_result['content']['sha']
print(f"✅ 文件更新成功 (new sha: {new_file_sha[:7]})")

✅ 文件更新成功 (new sha: ba6c4b8)


In [ ]:
# 9.12 从 feature-a 发起 PR
pr = client.create_pull_request(
    TEST_OWNER, TEST_REPO,
    title="[Test] 验证 PR 全流程",
    head=branch_a,
    base="main",
    body="自动创建的测试 PR,包含文件创建和更新. "
)
pr_number = pr['number']
print(f"✅ PR 创建成功: #{pr_number} - {pr['html_url']}")

✅ PR 创建成功: #3 - https://github.com/SingularGuyLeBorn/github-api-lab/pull/3


In [ ]:
# 9.13 读取 PR 详情 + 文件变更
pr_detail = client.get_pull(TEST_OWNER, TEST_REPO, pr_number)
print(f"PR #{pr_detail['number']}: {pr_detail['title']}")
print(f"状态: {pr_detail['state']} | Draft: {pr_detail.get('draft', False)}")
print(f"可合并: {pr_detail.get('mergeable')}")
print(f"合并状态: {pr_detail.get('mergeable_state')}")

pr_files = client.get_pull_request_files(TEST_OWNER, TEST_REPO, pr_number)
print(f"\n文件变更 ({len(pr_files)} 个):")
for f in pr_files:
    print(f"  {f['status']:12} {f['filename']} (+{f['additions']}/-{f['deletions']})")

PR #3: [Test] 验证 PR 全流程
状态: open | Draft: False
可合并: True
合并状态: clean

文件变更 (1 个):
  added        test-feature.md (+6/-0)


In [ ]:
# 9.14 PR Review(COMMENT)
# 注意：同一账号不能 APPROVE 自己的 PR,GitHub 会报 422
review = client.create_pull_request_review(
    TEST_OWNER, TEST_REPO, pr_number,
    event="COMMENT",
    body="API 测试通过,代码变更清晰. "
)
print(f"✅ PR Review 完成: {review['state']} (ID: {review['id']})")

✅ PR Review 完成: COMMENTED (ID: 4161311027)


In [ ]:
# 9.15 合并 PR
merge_result = client.merge_pull_request(
    TEST_OWNER, TEST_REPO, pr_number,
    merge_method="squash",
    commit_title=f"[Test] Squash PR #{pr_number}",
    commit_message="由 API 测试自动合并. "
)
print(f"✅ PR 合并成功: {merge_result['sha'][:7]}")

✅ PR 合并成功: f12cb07


In [ ]:
# 9.16 删除 feature-a,保留 feature-b
client.delete_branch(TEST_OWNER, TEST_REPO, branch=branch_a)
print(f"✅ 分支 {branch_a} 已删除")

print(f"ℹ️  分支 {branch_b} 保留在仓库中,可查看")

✅ 分支 feature-a 已删除
ℹ️  分支 feature-b 保留在仓库中,可查看


In [ ]:
# 9.17 最终验证
issue_a_final = client.get_issue(TEST_OWNER, TEST_REPO, issue_a_number)
issue_b_final = client.get_issue(TEST_OWNER, TEST_REPO, issue_b_number)
comments_a = client.list_issue_comments(TEST_OWNER, TEST_REPO, issue_a_number)
comments_b = client.list_issue_comments(TEST_OWNER, TEST_REPO, issue_b_number)

print("=" * 50)
print(f"Issue A: #{issue_a_final['number']} [{issue_a_final['state'].upper()}]")
print(f"  评论数: {len(comments_a)}")
print(f"  → CLOSED 证明关闭功能成功")

print("-" * 50)
print(f"Issue B: #{issue_b_final['number']} [{issue_b_final['state'].upper()}]")
print(f"  评论数: {len(comments_b)}(含复杂 Markdown)")
print(f"  → OPEN 证明创建和重开功能成功")
print("=" * 50)

# 分支验证
branches = client.list_branches(TEST_OWNER, TEST_REPO, per_page=10)
branch_names = [b['name'] for b in branches]
print(f"\n当前分支: {branch_names}")
if branch_b in branch_names:
    print(f"✅ 保留分支 {branch_b} 仍在仓库中")


Issue A: #1 [CLOSED]
  评论数: 2
  → CLOSED 证明关闭功能成功
--------------------------------------------------
Issue B: #2 [OPEN]
  评论数: 2(含复杂 Markdown)
  → OPEN 证明创建和重开功能成功

当前分支: ['feature-b', 'main']
✅ 保留分支 feature-b 仍在仓库中


In [ ]:
# 9.18 (可选)删除测试仓库 —— 取消注释后执行

client.delete_repo(TEST_OWNER, TEST_REPO)
print("✅ 测试仓库已删除")
print("仓库保留. 可手动访问查看所有操作结果. ")

仓库保留. 可手动访问查看所有操作结果. 


In [ ]:
# 9.19a 修改仓库可见性
# gh> gh repo edit TEST_OWNER/TEST_REPO --visibility private
client.update_repo(TEST_OWNER, TEST_REPO, visibility="private")
print("✅ 仓库已设为 Private")


✅ 仓库已设为 Private
✅ 仓库已恢复为 Public


In [ ]:
# 9.19b 修改仓库可见性

# 再改回 Public,方便查看
client.update_repo(TEST_OWNER, TEST_REPO, visibility="public")
print("✅ 仓库已恢复为 Public")

### 9.20 高级能力验证(无写入)

分页、GraphQL、速率限制查询. 

In [ ]:
# 9.20a 分页获取 Issues
all_issues = client.paginate(
    f"/repos/{TEST_OWNER}/{TEST_REPO}/issues",
    params={"state": "all", "per_page": 10},
    max_pages=2
)
print(f"分页获取到 {len(all_issues)} 条 Issues")

In [ ]:
# 9.20b GraphQL 查询
result = client.graphql(
    """
    query($owner: String!, $repo: String!) {
      repository(owner: $owner, name: $repo) {
        stargazerCount
        forkCount
        issues(states: OPEN) { totalCount }
        pullRequests(states: MERGED) { totalCount }
      }
    }
    """,
    variables={"owner": TEST_OWNER, "repo": TEST_REPO}
)
r = result['data']['repository']
print(f"⭐ {r['stargazerCount']}  🍴 {r['forkCount']}  🐛 {r['issues']['totalCount']}  🔀 {r['pullRequests']['totalCount']}")

In [ ]:
# 9.20c 查询速率限制
rate = client.get_rate_limit()
core = rate.get('resources', {}).get('core', {})
print(f"核心 API 配额: {core.get('remaining')}/{core.get('limit')} (reset: {core.get('reset')})")

## 8. 关键发现汇总

运行完以上 Cell 后,把关键发现记录在这里：

| 能力 | API 端点 | gh CLI 等效 | 测试结果 | 备注 |
|------|---------|------------|---------|------|
| 获取仓库信息 | `GET /repos/{o}/{r}` | `gh repo view` | | |
| 搜索仓库 | `GET /search/repositories` | `gh search repos` | | |
| 列出目录 | `GET /repos/{o}/{r}/contents/{p}` | `gh api .../contents` | | |
| 读取文件 | `GET /repos/{o}/{r}/contents/{p}` | `gh api .([^0-9]).

| base64 -d` | | 需 base64 解码 |
| 列出 Issues | `GET /repos/{o}/{r}/issues` | `gh issue list` | | |
| 获取 Issue | `GET /repos/{o}/{r}/issues/{n}` | `gh issue view` | | |
| 列出 PRs | `GET /repos/{o}/{r}/pulls` | `gh pr list` | | |
| 获取 PR | `GET /repos/{o}/{r}/pulls/{n}` | `gh pr view` | | |
| 搜索代码 | `GET /search/code` | `gh search code` | | 限 10 req/min |
| 提交历史 | `GET /repos/{o}/{r}/commits` | `gh api .../commits` | | |
| 列出工作流 | `GET /repos/{o}/{r}/actions/workflows` | `gh workflow list` | | |
| 创建 Issue | `POST /repos/{o}/{r}/issues` | `gh issue create` | | 需 write 权限 |
| 更新 Issue | `PATCH /repos/{o}/{r}/issues/{n}` | `gh issue edit` | | 可关闭/改标签 |
| 创建/更新文件 | `PUT /repos/{o}/{r}/contents/{p}` | `gh api .../contents` | | 需 base64 编码 |
| 删除文件 | `DELETE /repos/{o}/{r}/contents/{p}` | `gh api .../contents` | | 需 sha |
| 创建 PR | `POST /repos/{o}/{r}/pulls` | `gh pr create` | | 需 write 权限 |
| 合并 PR | `PUT /repos/{o}/{r}/pulls/{n}/merge` | `gh pr merge` | | 支持 squash/rebase |
| PR Review | `POST /repos/{o}/{r}/pulls/{n}/reviews` | `gh pr review` | | |
| 创建分支 | `POST /repos/{o}/{r}/git/refs` | `gh api .../git/refs` | | 基于现有分支 |
| 删除分支 | `DELETE /repos/{o}/{r}/git/refs/heads/...` | `gh api .../git/refs` | | |
| 创建 Label | `POST /repos/{o}/{r}/labels` | `gh label create` | | |
| 创建 Release | `POST /repos/{o}/{r}/releases` | `gh release create` | | draft 不创建 tag |
| 分页查询 | Link header 翻页 | `gh ... --paginate` | | 自动拼接多页 |
| GraphQL 查询 | `POST /graphql` | `gh api graphql` | | 关联查询更高效 |
| 查询速率限制 | `GET /rate_limit` | `gh api rate_limit` | | |
| 运行记录 | `GET /repos/{o}/{r}/actions/runs` | `gh run list` | | |

### 速率限制观察

- 认证用户: 5000 req/hour
- 搜索 API: 10 req/minute(认证用户)
- 未认证: 60 req/hour

### Agent 工具设计建议

1. **必须带 GITHUB_TOKEN**：未认证速率太低(60/h),搜索 API 完全不可用
2. **前端直接调用 API**：GitHub REST API 无 CORS,前端可直接 fetch
3. **base64 解码在前端做**：`atob()` 即可解码文件内容
4. **搜索 API 独立限流**：需在前端做搜索调用频率控制
5. **Issue/PR 创建需要 write 权限**：Token 需 `repo` scope